# 01 - 使用 YOLOv8 进行人脸检测训练（WIDER FACE）

本 Notebook 覆盖：
1. 检查数据集路径
2. 运行数据切分脚本
3. 加载 YOLOv8n 模型
4. 训练与验证
5. 样例测试与可视化
6. 保存最佳模型到 `models/best.pt`

In [ ]:
# 中文注释：导入训练所需的常用库
from pathlib import Path
import shutil
import subprocess

import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

In [ ]:
# 中文注释：定义关键路径
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
source_dataset = project_root / 'data' / 'wider_face_yolo'
split_dataset = project_root / 'data' / 'wider_face_yolo_split'
face_yaml = project_root / 'data' / 'face.yaml'

print('project_root =', project_root)
print('source_dataset exists =', source_dataset.exists())
print('images exists =', (source_dataset / 'images').exists())
print('labels exists =', (source_dataset / 'labels').exists())
print('face_yaml exists =', face_yaml.exists())

In [ ]:
# 中文注释：运行数据切分脚本（80/10/10）
cmd = [
    'python',
    str(project_root / 'scripts' / 'split_dataset.py'),
    '--source', str(source_dataset),
    '--target', str(split_dataset),
    '--train-ratio', '0.8',
    '--val-ratio', '0.1',
    '--test-ratio', '0.1',
    '--seed', '42',
    '--copy'
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('数据切分失败，请检查路径和标签文件。')

In [ ]:
# 中文注释：加载 YOLOv8n 预训练模型
model = YOLO('yolov8n.pt')
print('模型加载成功。')

In [ ]:
# 中文注释：开始训练（可根据本机性能调整 epochs、imgsz、batch）
train_results = model.train(
    data=str(face_yaml),
    epochs=20,
    imgsz=640,
    batch=16,
    project=str(project_root / 'runs'),
    name='face_yolov8n',
)
print('训练完成。')

In [ ]:
# 中文注释：验证模型
val_results = model.val(data=str(face_yaml))
print(val_results)

In [ ]:
# 中文注释：在测试集上抽样推理并可视化
sample_images = list((split_dataset / 'images' / 'test').rglob('*.jpg'))[:4]
if not sample_images:
    sample_images = list((split_dataset / 'images' / 'test').rglob('*.png'))[:4]

if not sample_images:
    raise RuntimeError('测试集中没有可用图片，请确认数据切分是否成功。')

pred_results = model.predict([str(p) for p in sample_images], conf=0.25)

plt.figure(figsize=(12, 8))
for i, r in enumerate(pred_results, start=1):
    annotated = r.plot()[:, :, ::-1]  # BGR 转 RGB
    plt.subplot(2, 2, i)
    plt.imshow(annotated)
    plt.axis('off')
    plt.title(Path(r.path).name)

plt.tight_layout()
plt.show()

In [ ]:
# 中文注释：将最佳模型复制到 models/best.pt
best_src = project_root / 'runs' / 'face_yolov8n' / 'weights' / 'best.pt'
best_dst = project_root / 'models' / 'best.pt'

if not best_src.exists():
    raise FileNotFoundError(f'未找到训练输出模型: {best_src}')

best_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_src, best_dst)
print(f'最佳模型已保存到: {best_dst}')